# Shift-Scan: Real FineWeb-Edu Ordered Decay-Trace Study

**This notebook uses real text and a real pretrained model. It does not use generated or synthetic data for its empirical measurements.**

It loads the unquantized pretrained [Mamba-2 130M checkpoint](https://huggingface.co/state-spaces/mamba2-130m) and streams the [FineWeb-Edu `sample-10BT` dataset](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu). For each input sequence, it records the actual ordered continuous decay coefficients computed inside every Mamba-2 layer and head.

It then compares three one-term power-of-two methods on the *same recorded traces*:

1. **Nearest:** independent nearest exponent rounding.
2. **HARPQ:** the project's existing per-token stationary horizon-aware objective.
3. **Seq-CD:** sequence-aware coordinate refinement minimizing exact nonstationary products across observed contiguous windows.

The sequence-aware coordinate method is an **offline diagnostic** because it examines the full trace. It is not yet a causal decoding algorithm. This notebook measures retention approximation, not perplexity or speed.

> **Do not run this on the same busy Kaggle T4 session as the 20M-token QAT training cell.** Use a separate GPU session, or choose CPU mode (slower). This experiment does not need the student checkpoint and will not modify it.

## 1. Runtime and package setup

Recommended: Kaggle with Internet enabled and a T4 GPU. This script leaves the runtime's existing PyTorch/CUDA build untouched and installs only data/model helper libraries.

In [ ]:
import os, sys, platform, subprocess, importlib.util
print("Python:", platform.python_version())
print("Torch already installed:", importlib.util.find_spec("torch") is not None)
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("Install a PyTorch build compatible with this runtime before continuing.")
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Do not upgrade torch here; retain the platform's preinstalled CUDA build.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "datasets>=2.18", "transformers>=4.40", "huggingface_hub>=0.23", "safetensors>=0.4", "numpy>=1.24", "pandas>=1.5", "matplotlib>=3.6"
])
print("Dependencies installed. Restart only if the runtime explicitly requests it.")


## 2. Add the research pack

Download `ShiftScan_Real_FineWeb_Trace_Pack.zip` from this conversation and make it available in this notebook runtime.

- **Kaggle:** add the ZIP as a notebook input (or put it in `/kaggle/working`).
- **Colab:** the next cell offers a file-upload prompt if the ZIP is not already in `/content`.
- **Local Jupyter:** place the ZIP beside this notebook.

The pack contains the capture/analysis scripts plus the exact `ssm_core.py` and `retention.py` files copied from the previously uploaded Shift-Scan project source.

In [ ]:
from pathlib import Path
import glob, zipfile, os

if Path("/kaggle/working").exists():
    WORK = Path("/kaggle/working")
elif Path("/content").exists():
    WORK = Path("/content")
else:
    WORK = Path.cwd()

search_roots = [WORK, Path.cwd()]
if Path("/kaggle/input").exists():
    search_roots.append(Path("/kaggle/input"))
if Path("/content").exists():
    search_roots.append(Path("/content"))

candidates = []
for base in search_roots:
    candidates.extend(base.glob("ShiftScan_Real_FineWeb_Trace_Pack.zip"))
    if base.exists():
        candidates.extend(base.glob("**/ShiftScan_Real_FineWeb_Trace_Pack.zip"))

if not candidates and "google.colab" in sys.modules:
    from google.colab import files
    uploaded = files.upload()
    candidates = [Path(WORK / name) for name in uploaded if name.endswith(".zip")]

if not candidates:
    raise FileNotFoundError(
        "Research pack ZIP not found. Download ShiftScan_Real_FineWeb_Trace_Pack.zip from ChatGPT "
        "and attach it as a Kaggle input or place it in /kaggle/working (or /content)."
    )

PACK = candidates[0]
PACK_ROOT = WORK / "shift_scan_real_fineweb_pack"
PACK_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACK) as zf:
    bad = [n for n in zf.namelist() if Path(n).is_absolute() or ".." in Path(n).parts]
    if bad:
        raise RuntimeError(f"Unsafe ZIP paths found: {bad[:3]}")
    zf.extractall(PACK_ROOT)

CAPTURE = PACK_ROOT / "capture_real_decay_traces.py"
ANALYZE = PACK_ROOT / "analyze_real_decay_traces.py"
SOURCE = PACK_ROOT / "shift_scan_source"
assert CAPTURE.exists() and ANALYZE.exists() and (SOURCE / "ssm_core.py").exists()
print("Using pack:", PACK)
print("Extracted to:", PACK_ROOT)
print("Scripts and project source found.")


## 3. Configure and capture actual model decays

The default pilot captures 64 sequences × 1,024 tokens from FineWeb-Edu `sample-10BT`. All 24 layers and all heads are retained in float32, with time order preserved. The dataset document subsampling rule is deterministic (`document_index % 25 == 0`), and the exact resolved dataset revision plus model checkpoint SHA-256 are written to metadata.

The pretrained Mamba-2 model is run with its original exponential decay (`QuantConfig(decay="exp")`). No HARPQ/ternary quantization is applied during trace collection. The script checks checkpoint loading, tokenizer/model token-ID compatibility, and trace finiteness.

In [ ]:
# Adjust these only deliberately and keep the values with your experiment records.
DEVICE = "auto"       # "auto", "cuda", or "cpu"
SEQ_LEN = 1024
MAX_SEQUENCES = 64    # start with 64; increase to 128/256 for stronger statistics
VAL_MODULO = 25       # use every document index divisible by 25; set 1 to use every document
AMP_DTYPE = "fp16"    # GPU only; CPU capture runs FP32

TRACE_FILE = WORK / "real_fineweb_mamba2_decay_traces.npz"
META_FILE = WORK / "real_fineweb_mamba2_decay_traces.meta.json"

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA requested but unavailable. Select DEVICE='cpu' or enable a GPU.")
if DEVICE == "auto" and torch.cuda.is_available():
    print("Capture will use GPU:", torch.cuda.get_device_name(0))
else:
    print("Capture device:", "CPU" if DEVICE == "cpu" or not torch.cuda.is_available() else DEVICE)

print("IMPORTANT: If this runtime shares a GPU with your active QAT run, stop here and use a separate session.")


In [ ]:
cmd = [
    sys.executable, str(CAPTURE),
    "--project-dir", str(SOURCE),
    "--seq-len", str(SEQ_LEN), "--chunk", "64",
    "--max-sequences", str(MAX_SEQUENCES), "--val-modulo", str(VAL_MODULO),
    "--dtype", AMP_DTYPE, "--device", DEVICE,
    "--out", str(TRACE_FILE), "--meta-out", str(META_FILE),
]
print("Running:\n", " ".join(cmd))
subprocess.run(cmd, check=True)
assert TRACE_FILE.exists() and META_FILE.exists()
print("Trace bytes:", TRACE_FILE.stat().st_size)
print("Metadata bytes:", META_FILE.stat().st_size)


## 4. Verify the captured trace artifact

This cell verifies the saved array shape, dtype, value range, number of tokens, and metadata before any quantizer comparison. If this cell fails, do not use the trace results.

In [ ]:
import json, numpy as np
with np.load(TRACE_FILE, allow_pickle=False) as z:
    decay_traces = z["decay_ref"]
    doc_spans = z["source_document_spans"]
meta = json.loads(META_FILE.read_text())
print("Trace shape [sequence, layer, time, head]:", decay_traces.shape)
print("Trace dtype:", decay_traces.dtype)
print("Source-document spans shape:", doc_spans.shape)
print("Min / max decay:", float(decay_traces.min()), float(decay_traces.max()))
print("All finite:", bool(np.isfinite(decay_traces).all()))
print("Captured tokens:", meta["capture"]["tokens_seen_by_model"])
print("Model revision:", meta["model"]["revision"])
print("Model checkpoint SHA256:", meta["model"]["checkpoint_sha256"])
print("Dataset revision:", meta["source"]["dataset_revision"])
assert decay_traces.ndim == 4 and decay_traces.shape[0] == MAX_SEQUENCES
assert np.isfinite(decay_traces).all() and np.all(decay_traces > 0) and np.all(decay_traces < 1)
assert decay_traces.dtype == np.float32


## 5. Compare nearest, existing HARPQ, and sequence-aware refinement

The sequence-aware refinement optimizes a genuinely nonstationary objective on observed decay windows. It does not replace `a_t` by `a_t**lag` as if a token's decay stayed constant. The initial pilot below analyzes 24 independent captured sequences, one randomly chosen layer/head trace per sequence, the first 256 positions, and all lags through 128. It uses one coordinate pass to bound runtime. Increase the sample and passes for a stronger follow-up.

In [ ]:
ANALYSIS_DIR = WORK / "real_fineweb_trace_analysis"
cmd = [
    sys.executable, str(ANALYZE),
    "--traces", str(TRACE_FILE), "--meta", str(META_FILE),
    "--out-dir", str(ANALYSIS_DIR),
    "--k-min", "1", "--k-max", "14", "--horizon", "128",
    "--analysis-length", "256", "--analysis-sequences", "24",
    "--traces-per-sequence", "1", "--passes", "1", "--seed", "123",
]
print("Running:\n", " ".join(cmd))
subprocess.run(cmd, check=True)
SUMMARY_FILE = ANALYSIS_DIR / "summary.json"
TRACE_METRICS_FILE = ANALYSIS_DIR / "per_trace_metrics.csv"
assert SUMMARY_FILE.exists() and TRACE_METRICS_FILE.exists()


In [ ]:
import pandas as pd
summary = json.loads(SUMMARY_FILE.read_text())
metrics_df = pd.read_csv(TRACE_METRICS_FILE)
print("Aggregate by sequence bootstrap (95% CI):")
for metric, values in summary["aggregate_by_sequence_bootstrap_95ci"].items():
    print(metric, values)
print("\nRelative retention-RMSE reduction vs nearest:")
print(summary["mean_relative_retention_rmse_reduction_vs_nearest"])
metrics_df.head(10)


In [ ]:
import matplotlib.pyplot as plt
cols = [
    "nearest_retention_rmse",
    "harpq_local_stationary_retention_rmse",
    "sequence_aware_coordinate_retention_rmse",
]
labels = ["Nearest", "HARPQ local", "Sequence-aware CD"]
plt.figure(figsize=(8, 4.5))
plt.boxplot([metrics_df[c].values for c in cols], labels=labels, showfliers=False)
plt.ylabel("Exact nonstationary retention RMSE")
plt.title("Real Mamba-2 decay traces on FineWeb-Edu")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## 6. Save and export the real-data artifacts

Keep these files together: trace NPZ, capture metadata, summary JSON, per-trace CSV, logs, and this notebook. Do not mix these diagnostic metrics with student perplexity or the 20M-token QAT training logs.

In [ ]:
import shutil
EXPORT = WORK / "ShiftScan_Real_FineWeb_Trace_Results.zip"
with zipfile.ZipFile(EXPORT, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in [TRACE_FILE, META_FILE, SUMMARY_FILE, TRACE_METRICS_FILE]:
        zf.write(path, arcname=path.name if path.parent == WORK else f"analysis/{path.name}")
print("Export ready:", EXPORT, "bytes:", EXPORT.stat().st_size)

# In Colab this offers a direct download. In Kaggle, use the Output panel to save/download the ZIP.
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(EXPORT))


## 7. What these results can and cannot support

If Seq-CD lowers exact nonstationary retention RMSE on these real traces, that supports only the narrow statement that sequence-aware exponent assignment improves decay-product approximation on this sampled model/data slice. Because Seq-CD uses the full sequence, it is a retrospective/offline diagnostic. It does not by itself demonstrate a causal online algorithm, lower perplexity, faster kernels, lower energy, or a better trained student.

For a stronger paper result, repeat the capture with more FineWeb-Edu sequences and at least one independent seed/shard selection; bootstrap by document/sequence groups; inspect layer/head heterogeneity; and validate any deployable causal variant separately. The next downstream experiment would be a matched QAT comparison using identical initialization, token budget, sequence length, precision, optimizer, and evaluation protocol.